# rlmflow coding-agent walkthrough

Build a recursive coding agent, run a task, save the trace, and embed the generated artifact inline. This mirrors `examples/coding-agent/agent.py` but as a notebook so you can poke at every step.

Sibling notebooks read offline against the deterministic fixture under `examples/data/notebook-coding-agent/` (regenerated by `examples/data/build_notebook_fixture.py`):

- [`node_basics.ipynb`](./node_basics.ipynb) — querying the trace: `graph.walk`, `graph.find`, `graph.where`, `session.load_graph`, …
- [`viz_walkthrough.ipynb`](./viz_walkthrough.ipynb) — visualizing the trace: `tree`, `gantt`, `code_log`, mermaid / dot / d2, `report_md`, the inline Gradio viewer, etc.

Running this notebook end-to-end requires `OPENAI_API_KEY` and a live LLM call. Skip ahead to the other notebooks if you just want to consume the saved trace.

## 1. Build the agent

Four pieces snap together:

- `Workspace` — durable, branchable storage for sessions, traces, and any files the agent writes.
- `Runtime` — where REPL code runs. `LocalRuntime` runs in-process; `DockerRuntime` runs each step in a container.
- Tools — plain Python functions registered on the runtime. `FILE_TOOLS` gives the agent `read_file`, `write_file`, `edit_file`, `ls`, `grep`, etc.
- `RLMFlow` — wires an LLM client (plus optional cheaper alternates) to the runtime + workspace, and exposes `start` / `step` / `fork`. Every call to `start` / `step` returns a fresh immutable `Graph`.

In [1]:
from pathlib import Path

from rlmflow.llm import OpenAIClient
from rlmflow.prompts.default import DEFAULT_BUILDER
from rlmflow.rlm import RLMConfig, RLMFlow
from rlmflow.runtime.local import LocalRuntime
from rlmflow.tools import FILE_TOOLS
from rlmflow.workspace import Workspace

def build_agent(
    workspace_dir: str | Path = "./notebook-coding-agent",
    max_depth: int = 3,
    max_iterations: int = 30,
    max_concurrency: int = 8,
) -> RLMFlow:
    """Construct a coding agent identical to examples/coding-agent/agent.py."""
    workspace = Workspace.create(Path(workspace_dir).resolve())
    runtime = LocalRuntime(workspace=workspace)
    runtime.register_tools(FILE_TOOLS)

    return RLMFlow(
        llm_client=OpenAIClient("gpt-5-mini"),
        runtime=runtime,
        workspace=workspace,
        config=RLMConfig(
            max_depth=max_depth,
            max_iterations=max_iterations,
            max_concurrency=max_concurrency,
        ),
    )

## 2. Run a task

`agent.start(query)` returns the initial `Graph` (just the root agent with its `QueryNode`). Every `agent.step(graph)` advances exactly one tick — one LLM call, one REPL execution, or a wait-resolution — and returns a freshly-loaded `Graph` snapshot. Use `graph.current()` to inspect the latest state, `graph.finished` to stop, `graph.result()` for the terminal answer.

`live_view()` opens a self-updating Rich tree; you own the loop and just hand it the latest `Graph` on each tick. The agent loop and the visualization are decoupled.

In [2]:
TASK = """Create a runnable boids simulation in plain HTML, CSS, and JavaScript. It should render fast-moving and colorful boids on a 2D canvas.

Requirements:
- The main runnable interface is `index.html`.
- Split the implementation into separate components and files. 
- Verify that all files were created and wired together before returning done, with no syntax errors.
"""

agent = build_agent(max_depth=2, workspace_dir="./boids-sim-workspace")
graph = agent.start(TASK)
graphs = [graph]

In [3]:
print(agent.build_system_prompt(graph))

## Role

You are answering a query with associated context in an iterative Python REPL.

## REPL

- Reply with exactly one fenced ` ```repl ` code block. Tools are already in the namespace.
- Think step by step in code/comments, plan briefly, then execute immediately.
- Output is truncated, so keep large data in variables/buffers and print summaries.
- Use Python to inspect, compute, branch, delegate, aggregate, and verify.
- Use only functions/tools that are actually present in the REPL or listed under `Tools`.

## Strategy

Inspect -> decompose -> batch -> wait -> verify -> done.

- Use the REPL as the work surface: inspect, compute, delegate, aggregate.
- If work splits into independent units, the parent fans out by default: chunks, documents, files, paths, records, trials, checks, components, artifacts, subproblems.
- Parent pattern: define shared contract + unit scopes -> spawn all children -> `yield rlm_wait(*handles)` -> verify/synthesize.
- Do not draft, generate, or write all 

In [4]:
from rlmflow.utils.viz import live_view

with live_view() as view:
    view(graph)
    while not graph.finished:
        graph = agent.step(graph)
        graphs.append(graph)
        view(graph)

Output()

In [58]:
current = graph.current()
print(f"{len(graphs)} snapshots  \u00b7  final: {graph.root_agent_id} [{current.type}]")
print(f"query : {graph.query[:120]!r}...")
print(f"result: {graph.result()[:200]}")

14 snapshots  ·  final: root [done_output]
query : 'Create a runnable boids simulation in plain HTML, CSS, and JavaScript. It should render fast-moving and colorful boids o'...
result: Created files:
- index.html (336 bytes)
- styles.css (115 bytes)
- js/boid.js (1543 bytes)
- js/flock.js (7511 bytes)
- js/renderer.js (3036 bytes)
- js/main.js (3433 bytes)


In [59]:
print(graph.tree())

● root (default) — Create a runnable boids simulation in plain HTML, CSS, and …
  - [ 0] query
  - [ 1] llm
  - [ 2] llm code=# Plan: # - Fan out children to create …
  - [ 3] exec
  - [ 4] supervising waiting_on=['root.index_html', 'root.styles_css', 'root.boid_js', 'root.flock_js', 'root.renderer_js', 'root.main_js']
    ● root.index_html (default) — Goal: Create one component of a runnable Boids simulation f…
      - [ 0] query
      - [ 1] llm
      - [ 2] llm code=# Inspect context, then write the requi…
      - [ 3] exec
      - [ 4] done -> WROTE index.html (336 bytes)
    ● root.styles_css (default) — Goal: Create one component of a runnable Boids simulation f…
      - [ 0] query
      - [ 1] llm
      - [ 2] llm code=# Plan: Minimal single-file write. Buil…
      - [ 3] exec
      - [ 4] done -> WROTE styles.css (115 bytes)
    ● root.boid_js (default) — Goal: Create one component of a runnable Boids simulation f…
      - [ 0] query
      - [ 1] llm
      - [ 2] llm code=# Pla

In [60]:
# from rlmflow.utils.trace import save_trace

# trace_path = save_trace(graphs, agent.workspace.trace_dir, metadata={"task": TASK})
# print(f"trace -> {trace_path}  ({len(graphs)} snapshots)")

## 3. Embed the generated artifact

Inline the `index.html` the agent produced (resolving CSS / ES-module imports / plain `<script src=>` tags) and render it via `<iframe srcdoc=...>` so it works inside Jupyter (which does not resolve relative iframe URLs).

In [61]:
import html as _html
import re
from IPython.display import HTML

live_candidates = sorted(agent.workspace.root.glob("./**/index.html"))
canonical_candidates = sorted(
    Path("../data/boids-sim-workspace").glob("**/index.html")
)
candidates = live_candidates if live_candidates else canonical_candidates
if not candidates:
    raise FileNotFoundError(f"no index.html under {agent.workspace.root}")
html_path = candidates[-1]
base = html_path.parent


def _inline_css(m):
    p = base / m.group(1)
    return f"<style>\n{p.read_text()}\n</style>" if p.exists() else m.group(0)


def _inline_plain_js(m):
    p = base / m.group(1)
    return f"<script>\n{p.read_text()}\n</script>" if p.exists() else m.group(0)


def _flatten_module(entry: str) -> str:
    visited: set = set()
    chunks: list[str] = []

    def visit(path):
        path = path.resolve()
        if path in visited or not path.exists():
            return
        visited.add(path)
        src = path.read_text()
        for m in re.finditer(
            r"^\s*import\b.*?\bfrom\s+['\"]([^'\"]+)['\"]",
            src,
            flags=re.MULTILINE,
        ):
            visit(path.parent / m.group(1))
        src = re.sub(
            r"^\s*import\b.*?\bfrom\s+['\"][^'\"]+['\"];?\s*$",
            "",
            src,
            flags=re.MULTILINE,
        )
        src = re.sub(r"^\s*export\s+default\s+", "", src, flags=re.MULTILINE)
        src = re.sub(r"^\s*export\s+", "", src, flags=re.MULTILINE)
        chunks.append(f"// === {path.name} ===\n{src}")

    visit(base / entry)
    return "<script>\n" + "\n".join(chunks) + "\n</script>"


content = html_path.read_text()
content = re.sub(
    r'<link\b[^>]*\bhref="([^"]+)"[^>]*/?>', _inline_css, content, flags=re.I
)
content = re.sub(
    r'<script\b(?=[^>]*\btype="module")[^>]*\bsrc="([^"]+)"[^>]*>\s*</script>',
    lambda m: _flatten_module(m.group(1)),
    content,
    flags=re.I,
)
content = re.sub(
    r'<script\b[^>]*\bsrc="([^"]+)"[^>]*>\s*</script>',
    _inline_plain_js,
    content,
    flags=re.I,
)

print(f"rendering {html_path}")
HTML(
    f'<iframe srcdoc="{_html.escape(content, quote=True)}" '
    f'width="100%" height="600" '
    f'style="border:1px solid #ddd; border-radius:6px; background:#0a0a0f"></iframe>'
)

rendering /Users/shyam/Code/rlmkit/examples/notebooks/boids-sim-workspace/index.html


## 4. Open the interactive viewer

`open_viewer(source)` boots a Gradio stepper (step slider + clickable graph + per-agent transcript) inline.

In [ ]:
from rlmflow.utils.viewer import open_viewer

open_viewer("./boids-sim-workspace", inline=True, height=720, quiet=True)

## 4. Render frames of each step

In [ ]:
# rlmflow render ./boids-sim-workspace \
#   --format steps \
#   --out ./boids-sim-workspace/frames \
#   --width 1600 \
#   --height 1200 \
#   --scale 2 \
#   --marker-mult 3.5 \
#   --text-mult 2.2

In [ ]:
from rlmflow.utils.viewer import save_steps

save_steps(
    "./boids-sim-workspace",
    "./boids-sim-workspace/frames",
    width=1600,
    height=1200,
    scale=2,
    marker_mult=3.5,
    text_mult=2.2,
)

## Next

- [`node_basics.ipynb`](./node_basics.ipynb) — walk the saved trace with the `Graph` query API.
- [`viz_walkthrough.ipynb`](./viz_walkthrough.ipynb) — inline Plotly, Mermaid, DOT, Gantt, report exports.